# GBTRegressor avec `spark.ml` -- exemple sur jeu de donnees jouet

Ce notebook illustre l'ensemble du pipeline de regression par arbres boostes
(Gradient Boosted Trees) sur un jeu de donnees synthetique de 2 000 observations.

Chaque observation represente une mesure instantanee d'une station de velos.
La **variable cible** est `taux_occupation` (proportion de bornettes occupees,
entre 0 et 1). Les **variables explicatives** sont :

| Feature | Type | Description |
|---------|------|-------------|
| `heure` | Entier (0-23) | Heure de la mesure |
| `temperature_c` | Reel | Temperature exterieure en degres Celsius |
| `est_pluie` | Booleen | Precipitation > 0.5 mm/h |
| `est_weekend` | Booleen | Samedi ou dimanche |

Le taux d'occupation suit une **loi connue** (formulee ci-dessous) :
le modele devra la retrouver sans la connaître.

```
taux = profil_horaire(heure)
       + effet_pluie      (-0.18 si pluie)
       + effet_weekend    (-0.15 si weekend)
       + effet_temperature (+0.008 par degre au-dessus de 15°C)
       + bruit gaussien   (sigma = 0.04)
```

ou `profil_horaire` est une fonction bimodale avec un pic le matin (8 h)
et un pic le soir (18 h), comme on l'observe sur les vrais reseaux Velib'.


---
## 0. Initialisation de la SparkSession


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("GBT-Jouet")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} pret.")


---
## 1. Generation du jeu de donnees jouet

On genere les donnees avec une formule connue. En pratique, cette formule
est evidemment inconnue -- c'est ce que le modele doit apprendre.


In [ ]:
import numpy as np
import pandas as pd

SEED = 42
rng  = np.random.default_rng(SEED)
N    = 2_000

# ── Features brutes ──────────────────────────────────────────────────────────
heure         = rng.integers(0, 24, N)
temperature_c = rng.uniform(0, 30, N)
est_pluie     = rng.binomial(1, 0.25, N)   # pluie 25 % du temps
est_weekend   = rng.binomial(1, 2/7,  N)   # 2 jours sur 7

# ── Formule du taux d'occupation ─────────────────────────────────────────────
def profil_horaire(h):
    """Courbe bimodale : pic a 8 h (navettes) et a 18 h (retours)."""
    matin = np.exp(-0.5 * ((h - 8)  / 2.0) ** 2) * 0.65
    soir  = np.exp(-0.5 * ((h - 18) / 2.0) ** 2) * 0.55
    nuit  = 0.08
    return nuit + matin + soir

base               = profil_horaire(heure)
effet_pluie_arr    = np.where(est_pluie,   -0.18, 0.0)
effet_weekend_arr  = np.where(est_weekend, -0.15, 0.0)
effet_temp_arr     = 0.008 * (temperature_c - 15)
bruit              = rng.normal(0, 0.04, N)

taux_occupation = np.clip(
    base + effet_pluie_arr + effet_weekend_arr + effet_temp_arr + bruit,
    0.0, 1.0
)

# ── Construction du DataFrame Pandas puis Spark ───────────────────────────────
df_pd = pd.DataFrame({
    "id"           : range(N),
    "heure"        : heure.astype(int),
    "temperature_c": temperature_c.round(2),
    "est_pluie"    : est_pluie.astype(bool),
    "est_weekend"  : est_weekend.astype(bool),
    "taux_occupation": taux_occupation.round(4),
})

df = spark.createDataFrame(df_pd)
df.printSchema()
df.show(8)

print(f"\n{N} observations  --  taux min={taux_occupation.min():.3f}  "
      f"max={taux_occupation.max():.3f}  moy={taux_occupation.mean():.3f}")


---
## 2. Exploration visuelle

Avant de modeliser, on visualise les relations entre les features et la cible.
Cette etape guide le choix des features et la detection des outliers.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# -- Profil horaire moyen --
ax1 = fig.add_subplot(gs[0, :2])
profil_moy = df_pd.groupby("heure")["taux_occupation"].mean()
heures_ref = np.linspace(0, 23, 300)
ax1.plot(heures_ref, profil_horaire(heures_ref), "--",
         color="#95a5a6", linewidth=1.5, label="Formule theorique")
ax1.bar(profil_moy.index, profil_moy.values,
        alpha=0.6, color="#3498db", label="Moyenne observee")
ax1.set_xlabel("Heure de la journee")
ax1.set_ylabel("Taux d'occupation moyen")
ax1.set_title("Profil horaire -- le modele devra retrouver cette courbe")
ax1.set_xticks(range(0, 24, 2))
ax1.legend(); ax1.grid(alpha=0.3)

# -- Effet de la pluie --
ax2 = fig.add_subplot(gs[0, 2])
df_pd.boxplot(column="taux_occupation", by="est_pluie", ax=ax2,
              boxprops=dict(color="#2c3e50"),
              medianprops=dict(color="#e74c3c", linewidth=2))
ax2.set_xticklabels(["Sec", "Pluie"])
ax2.set_xlabel("Condition meteorologique")
ax2.set_ylabel("Taux d'occupation")
ax2.set_title("Effet de la pluie")
plt.sca(ax2); plt.title("Effet de la pluie")

# -- Effet du weekend --
ax3 = fig.add_subplot(gs[1, 0])
df_pd.boxplot(column="taux_occupation", by="est_weekend", ax=ax3,
              boxprops=dict(color="#2c3e50"),
              medianprops=dict(color="#e74c3c", linewidth=2))
ax3.set_xticklabels(["Semaine", "Weekend"])
ax3.set_xlabel(""); ax3.set_ylabel("Taux d'occupation")
plt.sca(ax3); plt.title("Effet du weekend")

# -- Effet de la temperature --
ax4 = fig.add_subplot(gs[1, 1])
ax4.scatter(df_pd["temperature_c"], df_pd["taux_occupation"],
            alpha=0.15, s=12, color="#9b59b6")
z = np.polyfit(df_pd["temperature_c"], df_pd["taux_occupation"], 1)
ax4.plot(np.sort(df_pd["temperature_c"]),
         np.polyval(z, np.sort(df_pd["temperature_c"])),
         color="#e74c3c", linewidth=2, label=f"Tendance lineaire")
ax4.set_xlabel("Temperature (°C)"); ax4.set_ylabel("Taux d'occupation")
ax4.set_title("Effet de la temperature")
ax4.legend(); ax4.grid(alpha=0.3)

# -- Distribution de la cible --
ax5 = fig.add_subplot(gs[1, 2])
ax5.hist(df_pd["taux_occupation"], bins=40, color="#1abc9c", alpha=0.8, edgecolor="white")
ax5.set_xlabel("Taux d'occupation"); ax5.set_ylabel("Frequence")
ax5.set_title("Distribution de la cible")
ax5.grid(alpha=0.3)

fig.suptitle("Exploration du jeu de donnees jouet", fontsize=13, y=1.01)
plt.savefig("exploration.png", dpi=120, bbox_inches="tight")
plt.show()


---
## 3. Feature engineering

### Encodage cyclique de l'heure

L'heure est une variable cyclique : l'heure 23 et l'heure 0 sont adjacentes,
mais numeriquement eloignees. On les encode sur un cercle avec sin/cos :

```
heure_sin = sin(heure x 2π / 24)
heure_cos = cos(heure x 2π / 24)
```

Avec cet encodage, la distance euclidienne entre h=23 et h=0 est faible --
comme elle doit l'etre.


In [ ]:
from pyspark.sql import functions as F
import math

PI = math.pi

df_feat = (
    df
    .withColumn("heure_sin",      F.sin(F.col("heure") * (2 * PI / 24)))
    .withColumn("heure_cos",      F.cos(F.col("heure") * (2 * PI / 24)))
    .withColumn("est_pluie_int",  F.col("est_pluie").cast("int"))
    .withColumn("est_weekend_int",F.col("est_weekend").cast("int"))
)

FEATURES = ["heure_sin", "heure_cos", "temperature_c",
            "est_pluie_int", "est_weekend_int"]
CIBLE    = "taux_occupation"

df_feat.select(["heure", "heure_sin", "heure_cos",
                "est_pluie", "est_pluie_int",
                "taux_occupation"]).show(6, truncate=False)


In [ ]:
# Illustration : l'encodage sin/cos preserve la circularite
heures = list(range(24))
sins   = [round(math.sin(h * 2 * PI / 24), 3) for h in heures]
coss   = [round(math.cos(h * 2 * PI / 24), 3) for h in heures]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(heures, sins, "o-", label="heure_sin", color="#e74c3c")
ax.plot(heures, coss, "s-", label="heure_cos", color="#3498db")
ax.set_xlabel("Heure"); ax.set_ylabel("Valeur encodee")
ax.set_title("Encodage cyclique sin/cos -- h=23 et h=0 ont des valeurs proches")
ax.set_xticks(range(0, 24, 2)); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("encodage_cyclique.png", dpi=120, bbox_inches="tight")
plt.show()


---
## 4. Split train / test

Sur ce jeu de donnees jouet, les observations sont independantes
(pas de structure temporelle) -- on peut donc utiliser un split aleatoire.

> **Rappel important** : dans le projet ClimaCity, les observations sont
> des series temporelles. Le split doit alors etre **chronologique**
> (2022 = train, 2023 = test) pour eviter la fuite d'information.


In [ ]:
df_train, df_test = df_feat.randomSplit([0.8, 0.2], seed=SEED)

# Mise en cache : les deux splits seront accedes plusieurs fois
df_train.cache(); df_train.count()
df_test.cache();  df_test.count()

print(f"Train : {df_train.count():,} lignes ({df_train.count()/N:.0%})")
print(f"Test  : {df_test.count():,} lignes ({df_test.count()/N:.0%})")


---
## 5. Construction du Pipeline MLlib

```
DataFrame  -->  VectorAssembler  -->  StandardScaler  -->  GBTRegressor
               (5 colonnes          (centre et reduit     (entraine le
                -> 1 vecteur)        chaque feature)       modele)
```

### Pourquoi le StandardScaler pour le GBT ?

Le GBTRegressor est base sur des arbres de decision, qui comparent des features
par des seuils. Il est theoriquement **insensible a l'echelle** des features.
Normaliser ne change pas le pouvoir predictif du modele.

En pratique, on normalise quand meme pour deux raisons :

1. **Homogeneite du pipeline** : si on ajoute un jour un regresseur lineaire
   ou un SVM (tres sensibles a l'echelle), le pipeline reste valide sans modification.
2. **Interpretation des importances** : les importances de features mesurent
   la reduction d'impurete, pas les coefficients -- elles ne dependent pas de l'echelle.
   Mais avoir des features comparables facilite la lecture.


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# ── Etape 1 : VectorAssembler ─────────────────────────────────────────────────
assembler = VectorAssembler(
    inputCols=FEATURES,
    outputCol="features_brutes",
    handleInvalid="skip"
)

# ── Etape 2 : StandardScaler ──────────────────────────────────────────────────
scaler = StandardScaler(
    inputCol="features_brutes",
    outputCol="features",
    withMean=True,
    withStd=True
)

# ── Etape 3 : GBTRegressor ────────────────────────────────────────────────────
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=CIBLE,
    predictionCol="prediction",
    maxIter=80,        # nombre d'arbres
    maxDepth=4,        # profondeur de chaque arbre
    stepSize=0.1,      # taux d'apprentissage
    seed=SEED
)

# ── Pipeline ──────────────────────────────────────────────────────────────────
pipeline = Pipeline(stages=[assembler, scaler, gbt])

print("Pipeline construit :")
for i, s in enumerate(pipeline.getStages()):
    print(f"  Stage {i} : {type(s).__name__}")


---
## 6. Entrainement


In [ ]:
import time

print("Entrainement du pipeline GBT...")
t0    = time.perf_counter()
model = pipeline.fit(df_train)
t_fit = time.perf_counter() - t0

print(f"Termine en {t_fit:.1f} s")
print()
print("Stages du PipelineModel :")
for i, s in enumerate(model.stages):
    print(f"  Stage {i} : {type(s).__name__}")


In [ ]:
# Acces direct aux stages entraines
scaler_model = model.stages[1]   # StandardScalerModel
gbt_model    = model.stages[2]   # GBTRegressionModel

print(f"Moyennes apprises par le scaler : {scaler_model.mean.toArray().round(4)}")
print(f"Ecarts-types                    : {scaler_model.std.toArray().round(4)}")
print()
print(f"Nombre d'arbres construits  : {gbt_model.numTrees}")
print(f"Profondeur maximale reelle  : {max(t.depth for t in gbt_model.trees)}")


---
## 7. Evaluation

On calcule trois metriques sur le **test set** (jamais vu pendant l'entrainement) :

| Metrique | Signification |
|----------|---------------|
| **RMSE** | Ecart-type des erreurs. Penalise davantage les grosses erreurs. |
| **MAE** | Erreur absolue moyenne. Plus robuste aux outliers. |
| **R²** | Part de variance expliquee. 1 = parfait, 0 = modele nul. |


In [ ]:
# Application du modele sur le test set
df_pred = model.transform(df_test)

# Evaluateurs
def evaluer(df_p, label=CIBLE, pred="prediction"):
    ev = lambda m: RegressionEvaluator(
        labelCol=label, predictionCol=pred, metricName=m).evaluate(df_p)
    return {"RMSE": ev("rmse"), "MAE": ev("mae"), "R2": ev("r2")}

metriques = evaluer(df_pred)

print("Metriques sur le test set :")
for nom, val in metriques.items():
    print(f"  {nom:<6} : {val:.4f}")
print()
print(f"Interpretation :")
print(f"  Le modele se trompe en moyenne de ±{metriques['MAE']:.3f} sur le taux (0-1),")
print(f"  soit ±{metriques['MAE']*100:.1f} points de pourcentage.")
print(f"  Il explique {metriques['R2']:.1%} de la variance du taux d'occupation.")


In [ ]:
# ── Comparaison avec un modele de reference (baseline) ───────────────────────
# Le modele le plus simple possible : toujours predire la moyenne du train set.
# Si le GBT ne bat pas ce modele, quelque chose ne va pas.

mean_train = df_train.agg(F.mean(CIBLE)).first()[0]
df_baseline = df_pred.withColumn("pred_baseline", F.lit(mean_train))

ev_baseline = evaluer(df_baseline, pred="pred_baseline")

print("Comparaison GBT vs Baseline (predire la moyenne) :")
print(f"  {'Metrique':<8} {'Baseline':>12} {'GBT':>12} {'Gain':>10}")
print("  " + "-" * 46)
for m in ["RMSE", "MAE", "R2"]:
    gain = metriques[m] - ev_baseline[m]
    signe = "-" if m != "R2" else "+"
    print(f"  {m:<8} {ev_baseline[m]:>12.4f} {metriques[m]:>12.4f} "
          f"{abs(gain):>+10.4f}")
print()
print(f"La baseline a un R²={ev_baseline['R2']:.3f} (modele nul = 0 par definition).")
print(f"Le GBT atteint R²={metriques['R2']:.3f} -- gain substantiel.")


---
## 8. Importance des features

L'**importance d'une feature** dans un GBT mesure la reduction d'impurete
(variance residuelle) moyenne qu'elle apporte a tous les noeuds ou elle est
utilisee, ponderee par le nombre de points qui passent par ces noeuds.

Elle est normalisee pour que la somme soit 1.

Une feature peu importante n'est pas necessairement inutile -- elle peut etre
correllee a une feature plus importante. En revanche, une feature avec une
importance nulle peut etre retiree sans perte.


In [ ]:
import pandas as pd

importances = gbt_model.featureImportances.toArray()

df_imp = (
    pd.DataFrame({"feature": FEATURES, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Importance des features :")
print(f"  {'Rang':<5} {'Feature':<20} {'Importance':>12}  Barre")
print("  " + "-" * 55)
for _, row in df_imp.iterrows():
    barre = "█" * int(row["importance"] * 50)
    print(f"  {_+1:<5} {row['feature']:<20} {row['importance']:>12.4f}  {barre}")

print()
print("Verification : somme des importances =", round(importances.sum(), 6))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#e74c3c" if imp == df_imp["importance"].max() else "#3498db"
          for imp in df_imp["importance"]]
ax.barh(df_imp["feature"][::-1], df_imp["importance"][::-1],
        color=colors[::-1], alpha=0.85, edgecolor="white")
ax.set_xlabel("Importance (reduction d'impurete normalisee)")
ax.set_title("Importance des features -- GBTRegressor")
ax.grid(alpha=0.3, axis="x")

# Annotations
for i, (feat, imp) in enumerate(zip(df_imp["feature"][::-1], df_imp["importance"][::-1])):
    ax.text(imp + 0.005, i, f"{imp:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("importance_features.png", dpi=120, bbox_inches="tight")
plt.show()

print()
print("Les features cycliques heure_sin et heure_cos dominent : le profil")
print("horaire est de loin le meilleur predicteur du taux d'occupation.")
print("C'est coherent avec la formule de generation.")


---
## 9. Visualisation des predictions et analyse des residus

Trois graphiques complementaires pour diagnostiquer la qualite du modele :

1. **Predit vs Reel** : les points doivent etre alignes sur la diagonale y=x.
2. **Residus vs Predit** : les residus doivent etre centres en 0, sans structure.
3. **Erreur absolue par heure** : identifie les plages horaires difficiles a predire.


In [ ]:
df_viz = df_pred.select("heure", CIBLE, "prediction").toPandas()
df_viz["residu"] = df_viz["prediction"] - df_viz[CIBLE]
df_viz["erreur_abs"] = df_viz["residu"].abs()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# -- 1. Predit vs Reel --
ax = axes[0]
ax.scatter(df_viz[CIBLE], df_viz["prediction"],
           alpha=0.25, s=12, color="#3498db")
lim = [0, 1]
ax.plot(lim, lim, "r--", linewidth=1.5, label="Prediction parfaite")
ax.set_xlabel("Valeur reelle"); ax.set_ylabel("Valeur predite")
ax.set_title("Predit vs Reel")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.legend(); ax.grid(alpha=0.3)
# RMSE sur le graphique
ax.text(0.05, 0.92, f"RMSE = {metriques['RMSE']:.4f}\nR² = {metriques['R2']:.4f}",
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# -- 2. Residus vs Predit --
ax = axes[1]
ax.scatter(df_viz["prediction"], df_viz["residu"],
           alpha=0.25, s=12, color="#9b59b6")
ax.axhline(0, color="red", linewidth=1.5, linestyle="--")
ax.axhline( df_viz["residu"].std(), color="#95a5a6", linewidth=1, linestyle=":")
ax.axhline(-df_viz["residu"].std(), color="#95a5a6", linewidth=1, linestyle=":",
           label=f"±1σ = ±{df_viz['residu'].std():.3f}")
ax.set_xlabel("Valeur predite"); ax.set_ylabel("Residu (predit - reel)")
ax.set_title("Residus vs Predit")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# -- 3. Erreur absolue par heure --
ax = axes[2]
mae_par_heure = df_viz.groupby("heure")["erreur_abs"].mean()
ax.bar(mae_par_heure.index, mae_par_heure.values,
       color="#e67e22", alpha=0.8)
ax.set_xlabel("Heure de la journee"); ax.set_ylabel("MAE moyenne")
ax.set_title("Erreur absolue par heure")
ax.set_xticks(range(0, 24, 2)); ax.grid(alpha=0.3, axis="y")

plt.suptitle("Diagnostic du modele GBTRegressor", fontsize=13)
plt.tight_layout()
plt.savefig("diagnostic_gbt.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Pourquoi certaines heures sont-elles plus difficiles ?
print("MAE par heure (triee) :")
print(mae_par_heure.sort_values(ascending=False).head(8).round(4).to_string())
print()
print("Les heures avec la plus forte MAE correspondent generalement")
print("aux transitions rapides (montee du pic matin, descente du soir)")
print("ou aux plages avec le plus de variance intrinseque.")


---
## 10. Verification : le modele a-t-il appris les bons effets ?

On peut verifier que le modele a bien capture les effets de chaque feature
en comparant les predictions moyennes selon les conditions.


In [ ]:
# Effet de la pluie capture par le modele
df_pluie = df_pred.groupby("est_pluie").agg(
    F.round(F.mean(CIBLE),         4).alias("taux_reel_moy"),
    F.round(F.mean("prediction"),  4).alias("taux_predit_moy"),
).orderBy("est_pluie")

print("Effet de la pluie :")
df_pluie.show()
print(f"  Vrai effet de la pluie dans la formule : -0.18")
pred_sec   = df_pluie.filter(~F.col("est_pluie")).first()["taux_predit_moy"]
pred_pluie = df_pluie.filter( F.col("est_pluie")).first()["taux_predit_moy"]
print(f"  Effet capture par le modele : {pred_pluie - pred_sec:.3f}")

print()

# Effet du weekend
df_wkd = df_pred.groupby("est_weekend").agg(
    F.round(F.mean(CIBLE),         4).alias("taux_reel_moy"),
    F.round(F.mean("prediction"),  4).alias("taux_predit_moy"),
).orderBy("est_weekend")

print("Effet du weekend :")
df_wkd.show()
print(f"  Vrai effet dans la formule : -0.15")
pred_sem = df_wkd.filter(~F.col("est_weekend")).first()["taux_predit_moy"]
pred_wkd = df_wkd.filter( F.col("est_weekend")).first()["taux_predit_moy"]
print(f"  Effet capture par le modele : {pred_wkd - pred_sem:.3f}")


In [ ]:
# Profil horaire predit vs theorique
df_horaire = (
    df_pred
    .groupby("heure")
    .agg(
        F.round(F.mean(CIBLE),        4).alias("taux_reel_moy"),
        F.round(F.mean("prediction"), 4).alias("taux_predit_moy"),
    )
    .orderBy("heure")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 4))
heures_ref = np.linspace(0, 23, 300)
ax.plot(heures_ref, profil_horaire(heures_ref), "--",
        color="#95a5a6", linewidth=2, label="Formule theorique (inconnue du modele)")
ax.plot(df_horaire["heure"], df_horaire["taux_reel_moy"],
        "o-", color="#3498db", linewidth=1.5, label="Taux reel moyen")
ax.plot(df_horaire["heure"], df_horaire["taux_predit_moy"],
        "s--", color="#e74c3c", linewidth=1.5, label="Prediction GBT moyenne")
ax.set_xlabel("Heure"); ax.set_ylabel("Taux d'occupation moyen")
ax.set_title("Profil horaire : le GBT retrouve la courbe bimodale")
ax.set_xticks(range(0, 24, 2)); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("profil_horaire_predit.png", dpi=120, bbox_inches="tight")
plt.show()
print("Le GBT reproduit fidelement le double pic matin/soir sans jamais l'avoir vu.")


---
## 11. Validation croisee et selection d'hyperparametres

On cherche la meilleure combinaison d'hyperparametres parmi une grille
de 6 configurations (3 profondeurs x 2 taux d'apprentissage).

Le `CrossValidator` entraine et evalue chaque combinaison sur 3 folds,
soit 18 entrainements au total.


In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth,  [3, 4, 5])
    .addGrid(gbt.stepSize,  [0.05, 0.1])
    .build()
)
print(f"Grille : {len(param_grid)} combinaisons x 3 folds = {len(param_grid)*3} entrainements")

evaluator_rmse = RegressionEvaluator(
    labelCol=CIBLE, predictionCol="prediction", metricName="rmse"
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_rmse,
    numFolds=3,
    seed=SEED,
    parallelism=2
)

print("\nValidation croisee en cours...")
t0       = time.perf_counter()
cv_model = cv.fit(df_train)
print(f"Terminee en {time.perf_counter()-t0:.1f} s")


In [ ]:
# Tableau des resultats par combinaison
configs = [
    {p.name: v for p, v in zip(param_grid[0].keys(), combo.values())}
    for combo in param_grid
]
df_cv = pd.DataFrame(configs)
df_cv["rmse_cv"] = cv_model.avgMetrics
df_cv = df_cv.sort_values("rmse_cv").reset_index(drop=True)

print("Resultats de la validation croisee (tries par RMSE croissant) :")
print(df_cv.to_string(index=False))

# Meilleurs hyperparametres
best_gbt_stage = cv_model.bestModel.stages[-1]
print(f"\nMeilleur maxDepth : {best_gbt_stage.getMaxDepth()}")
print(f"Meilleur stepSize  : {best_gbt_stage.getStepSize()}")


In [ ]:
# Evaluation du meilleur modele sur le test set
df_pred_best = cv_model.bestModel.transform(df_test)
metriques_best = evaluer(df_pred_best)

print("Comparaison modele initial vs meilleur modele CV :")
print(f"  {'Metrique':<8} {'Initial':>12} {'CV Best':>12} {'Gain':>10}")
print("  " + "-" * 46)
for m in ["RMSE", "MAE", "R2"]:
    gain = metriques[m] - metriques_best[m]
    print(f"  {m:<8} {metriques[m]:>12.4f} {metriques_best[m]:>12.4f} {gain:>+10.4f}")
print()
print("Sur ce petit jeu de donnees, le gain du tuning est souvent marginal.")
print("Il devient plus significatif quand la grille est plus large")
print("et le jeu de donnees plus complexe.")


---
## 12. Sauvegarde et rechargement du Pipeline

Le `PipelineModel` complet (assembler + scaler + gbt) est sauvegarde
en une seule operation. Au rechargement, toutes les transformations
apprises (moyennes du scaler, poids des arbres GBT...) sont restaurees.


In [ ]:
from pyspark.ml import PipelineModel

chemin = "/tmp/gbt_jouet_pipeline"
model.write().overwrite().save(chemin)
print(f"Modele sauvegarde dans : {chemin}/")
print()

# Structure du repertoire sauvegarde
import os
for racine, dossiers, fichiers in os.walk(chemin):
    niveau = racine.replace(chemin, "").count(os.sep)
    if niveau > 2:
        continue    # on limite la profondeur d'affichage
    indent = "  " * niveau
    print(f"{indent}{os.path.basename(racine)}/")
    for f in fichiers[:2]:
        print(f"{indent}  {f}")


In [ ]:
# Rechargement et application sur de nouvelles donnees
model_recharge = PipelineModel.load(chemin)

# Nouvelles stations jamais vues : on veut predire leur taux d'occupation
nouvelles = spark.createDataFrame([
    (1, 8,  15.0, False, False),   # matin de semaine, beau temps
    (2, 8,  15.0, True,  False),   # matin de semaine, sous la pluie
    (3, 18, 20.0, False, True),    # soir de weekend, beau temps
    (4, 3,  5.0,  False, False),   # nuit en semaine, froid
], ["id", "heure", "temperature_c", "est_pluie", "est_weekend"])

# Le pipeline applique automatiquement toutes les transformations apprises
nouvelles_feat = (
    nouvelles
    .withColumn("heure_sin",      F.sin(F.col("heure") * (2 * PI / 24)))
    .withColumn("heure_cos",      F.cos(F.col("heure") * (2 * PI / 24)))
    .withColumn("est_pluie_int",  F.col("est_pluie").cast("int"))
    .withColumn("est_weekend_int",F.col("est_weekend").cast("int"))
)

df_nouvelles_pred = model_recharge.transform(nouvelles_feat)
(
    df_nouvelles_pred
    .select("id", "heure", "est_pluie", "est_weekend",
            F.round("prediction", 3).alias("taux_predit"))
    .show()
)


---
## Bilan

### Ce que cet exemple illustre

| Etape | Classe MLlib | Role |
|-------|-------------|------|
| Encodage cyclique | `withColumn` + `sin`/`cos` | Heure circularisee |
| Assemblage | `VectorAssembler` | n colonnes -> 1 vecteur |
| Normalisation | `StandardScaler` | Echelles uniformes |
| Regression | `GBTRegressor` | Modele non-lineaire par arbres boostes |
| Orchestration | `Pipeline` | Chaine sans fuite d'information |
| Evaluation | `RegressionEvaluator` | RMSE, MAE, R² |
| Tuning | `CrossValidator` + `ParamGridBuilder` | Selection d'hyperparametres |
| Persistance | `PipelineModel` | Sauvegarde et rechargement |

### Ce que le GBT sait bien faire

- Capturer des **relations non-lineaires** sans les specifier explicitement
  (il a retrouve la courbe bimodale de lui-meme).
- Gerer des features d'**echelles et de natures tres differentes**
  (numeriques, binaires, cycliques).
- Fournir une **importance des features** directement interpretable.
- Etre **robuste aux outliers** (les seuils des arbres ne sont pas affectes
  par les valeurs extremes, contrairement aux coefficients lineaires).

### Ce que le GBT ne sait pas bien faire

- **Extrapoler** : si on lui presente une temperature de 50°C (jamais vue
  en entrainement), il renverra la prediction de la temperature maximale
  qu'il a observee -- pas une extrapolation lineaire.
- **Intervalles de confiance** : le GBT fournit une prediction ponctuelle,
  pas une distribution. Pour des intervalles, utiliser le Quantile Regression
  Forests ou le Conformal Prediction.
- **Tres grands espaces de features** (millions de dimensions) : les arbres
  de decision ne scalent pas aussi bien que les modeles lineaires dans ce regime.

### Passage au projet ClimaCity

Dans le Jour 3, le pipeline est identique dans sa structure. La difference est :
- 15 features au lieu de 5 (dont des lags temporels et le cluster K-Means).
- La variable cible est le taux a t+4 (prediction dans le futur).
- Le split est **chronologique** (2022 = train, 2023 = test) et non aleatoire.
- Les resultats sont loggues dans MLflow pour comparaison des runs.


In [ ]:
spark.stop()
print("SparkSession arretee.")
